In [0]:
CATALOGO_ESQUEMA = "bronce" 

# 1. --- Carga de las Tablas como Spark DataFrames (DFs Distribuidos) ---
print(f"Cargando tablas desde el esquema '{CATALOGO_ESQUEMA}'...")

customers_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_customers_dataset")
orders_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_orders_dataset")
payments_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_order_payments_dataset")
order_items_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_order_items_dataset")
products_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.olist_products_dataset")
category_translation_spark_df = spark.table(f"{CATALOGO_ESQUEMA}.product_category_name_translation")

print(f"Tablas cargadas a Spark DataFrames. Ejemplo: Registros de Clientes: {customers_spark_df.count()}")

# 2. --- Conversión a DataFrames de Pandas ---
# ¡ADVERTENCIA! Esta operación mueve todos los datos a la memoria de un solo nodo (Driver). 
# Solo debe usarse si el volumen total de datos es manejable (ej. < 1 GB).
print("\nConvirtiendo DataFrames de Spark a DataFrames de Pandas (.toPandas())...")

customers_df = customers_spark_df.toPandas()
orders_df = orders_spark_df.toPandas()
payments_df = payments_spark_df.toPandas()
order_items_df = order_items_spark_df.toPandas()
products_df = products_spark_df.toPandas()
category_translation_df = category_translation_spark_df.toPandas()

print(f"Conversión completada. Ejemplo: Tamaño del DataFrame de Clientes en Pandas: {len(customers_df)} filas.")
print(f"Primeros 5 registros del DataFrame de Clientes en Pandas:\n{customers_df.head()}")

# 3. --- Lista de DataFrames de Pandas para Procesamiento y Limpieza ---
# Ahora la lista 'datasets' contiene DataFrames de Pandas, lo que permite usar métodos nativos de Pandas.
datasets_dfs_pandas = [
    customers_df, 
    orders_df, 
    payments_df, 
    order_items_df, 
    products_df, 
    category_translation_df
]


In [0]:
%skip
# Databricks notebook: silver_ingestion_olist

from datetime import datetime
from pyspark.sql import functions as F

# === 1️⃣ Crear base de datos Silver si no existe ===
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# === 2️⃣ Definir datasets de entrada (desde capa Bronze) ===
datasets = [
    ("bronce.olist_customers_dataset", "silver.olist_customers_dataset"),
    ("bronce.olist_order_items_dataset", "silver.olist_order_items_dataset"),
    ("bronce.olist_orders_dataset", "silver.olist_orders_dataset")
]

# ===  Tabla de logs de limpieza ===
spark.sql("""
CREATE TABLE IF NOT EXISTS silver.logs_limpieza (
    fecha_ejecucion TIMESTAMP,
    tabla_origen STRING,
    tabla_destino STRING,
    registros_antes BIGINT,
    registros_despues BIGINT,
    nulos_eliminados BIGINT,
    duplicados_eliminados BIGINT,
    estado STRING
)
USING delta
""")

# === 4️⃣ Función de limpieza general ===
def limpiar_dataset(df, tabla):
    """
    Aplica reglas básicas de limpieza , id faltantes, campos faltantes , duplicados en las tres  bases de datos , valores invalidos.
    """
    nulos_eliminados = 0
    duplicados_eliminados = 0

    # Reglas específicas según el dataset
    if "customers" in tabla:
        # Eliminar filas sin 'customer_id' o 'customer_unique_id'
        registros_antes = df.count()
        df = df.dropna(subset=["customer_id", "customer_unique_id"])
        nulos_eliminados = registros_antes - df.count()
        # Eliminar duplicados por cliente único
        registros_antes = df.count()
        df = df.dropDuplicates(["customer_unique_id"])
        duplicados_eliminados = registros_antes - df.count()

    elif "order_items" in tabla:
        # Eliminar filas con campos clave nulos
        registros_antes = df.count()
        df = df.dropna(subset=["order_id", "product_id", "seller_id"])
        nulos_eliminados = registros_antes - df.count()
        # Eliminar duplicados por order_id y product_id
        registros_antes = df.count()
        df = df.dropDuplicates(["order_id", "product_id"])
        duplicados_eliminados = registros_antes - df.count()

    elif "orders" in tabla:
        # Filtrar solo estados válidos de pedidos
        registros_antes = df.count()
        estados_validos = ["delivered", "shipped", "invoiced", "processing", "created", "approved"]
        df = df.filter(F.col("order_status").isin(estados_validos))
        # Eliminar nulos en order_id y timestamps
        registros_antes2 = df.count()
        df = df.dropna(subset=["order_id", "order_purchase_timestamp"])
        nulos_eliminados = registros_antes - registros_antes2
        # Quitar duplicados por order_id
        registros_antes3 = df.count()
        df = df.dropDuplicates(["order_id"])
        duplicados_eliminados = registros_antes2 - registros_antes3

    else:
        # Limpieza genérica: eliminar nulos y duplicados
        registros_antes = df.count()
        df = df.dropna()
        nulos_eliminados = registros_antes - df.count()
        registros_antes = df.count()
        df = df.dropDuplicates()
        duplicados_eliminados = registros_antes - df.count()

    return df, nulos_eliminados, duplicados_eliminados


# === 5️⃣ Bucle principal de limpieza ===
for tabla_origen, tabla_destino in datasets:
    print(f"⏳ Procesando limpieza de: {tabla_origen}")

    try:
        # Leer datos desde capa bronce
        df = spark.table(tabla_origen)
        registros_antes = df.count()

        # Aplicar limpieza según dataset
        df_limpio, nulos_eliminados, duplicados_eliminados = limpiar_dataset(df, tabla_origen)
        registros_despues = df_limpio.count()

        # Agregar columna de control
        df_limpio = df_limpio.withColumn("cleaning_timestamp", F.current_timestamp())

        # Guardar en capa Silver
        df_limpio.write.mode("overwrite").format("delta").saveAsTable(tabla_destino)

        # Log de la ejecución
        log = [(datetime.now(), tabla_origen, tabla_destino, registros_antes, registros_despues,
                nulos_eliminados, duplicados_eliminados, "OK")]
        log_df = spark.createDataFrame(log, [
            "fecha_ejecucion", "tabla_origen", "tabla_destino", "registros_antes",
            "registros_despues", "nulos_eliminados", "duplicados_eliminados", "estado"
        ])
        log_df.write.mode("append").format("delta").saveAsTable("silver.logs_limpieza")

        print(f"✅ Limpieza completada: {tabla_destino}")
        print(f"   Registros antes: {registros_antes}")
        print(f"   Registros después: {registros_despues}")
        print(f"   Nulos eliminados: {nulos_eliminados}")
        print(f"   Duplicados eliminados: {duplicados_eliminados}\n")

    except Exception as e:
        print(f"❌ Error al procesar {tabla_origen}: {e}")

        # Log de error
        log = [(datetime.now(), tabla_origen, tabla_destino, 0, 0, 0, 0, f"ERROR: {e}")]
        log_df = spark.createDataFrame(log, [
            "fecha_ejecucion", "tabla_origen", "tabla_destino", "registros_antes",
            "registros_despues", "nulos_eliminados", "duplicados_eliminados", "estado"
        ])
        log_df.write.mode("append").format("delta").saveAsTable("silver.logs_limpieza")
        continue

print("🏁 Proceso de limpieza completado para todos los datasets.")
